# V2 Audio Patch

Patches `train_features_v2.pt` and `test_features_v2.pt` that were generated with all-zero audio.

**What this does (and does NOT do):**
- **Does NOT** re-run CLIP or ImageBind vision — those are already correct in the stored files.
- **Does** re-run VGGish audio extraction → overwrites `z_aud` [N, 128]
- **Does** re-run ImageBind audio extraction → fuses with the stored `v_teacher` (= raw IB_vision) to produce the proper multimodal teacher: `normalize((IB_vision + IB_audio) / 2)`
- **Does** update `has_audio` [N, bool]

**Key insight:** When `has_audio=False` the stored `v_teacher[i]` equals the raw unnormalized `IB_vision[i]` (no normalization was applied in `build_teacher`). So it can be reused directly — no need to re-run the vision encoder.

**Dataset:** Uses `nyhuka/msrvtt` Kaggle dataset as video source (has audio intact).
Video naming is consistent: `video1071.mp4` in nyhuka == `video1071.mp4` in our feature files.

Checkpoints every 200 videos. Safe to restart if the kernel dies.

## Step 1: Install Dependencies

In [ ]:
!pip install -q git+https://github.com/facebookresearch/ImageBind.git
!pip install -q resampy soundfile tqdm
# VGGish via torch.hub (no separate install needed)

## Step 2: Imports & GPU Check

In [ ]:
import os
import json
import subprocess
import zipfile
import urllib.request
import shutil

import torch
import torch.nn.functional as F
from tqdm import tqdm

from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
import imagebind.data as ib_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Local temp dir for wav extraction — avoids /tmp permission issues
TMP_DIR = "./tmp_audio"
os.makedirs(TMP_DIR, exist_ok=True)

## Step 3: Locate nyhuka/msrvtt Video Directory

Add `nyhuka/msrvtt` as an input dataset in the Kaggle notebook UI before running this cell.
This cell auto-discovers the directory that contains the `.mp4` files.

In [ ]:
def find_video_dir(base="/kaggle/input/msrvtt"):
    """Walk input dirs until we find a folder containing .mp4 files."""
    for root, dirs, files in os.walk(base):
        mp4s = [f for f in files if f.endswith(".mp4")]
        if mp4s:
            print(f"Found {len(mp4s)} .mp4 files in: {root}")
            print(f"  Sample: {sorted(mp4s)[:5]}")
            return root
    return None


# Try the expected path first, then scan all of /kaggle/input
VIDEO_DIR = find_video_dir("/kaggle/input/msrvtt")
if VIDEO_DIR is None:
    print("Not found under /kaggle/input/msrvtt — scanning all of /kaggle/input ...")
    VIDEO_DIR = find_video_dir("/kaggle/input")

if VIDEO_DIR is None:
    raise RuntimeError("Could not locate .mp4 files. Did you add nyhuka/msrvtt as an input dataset?")

print(f"\nVIDEO_DIR = {VIDEO_DIR}")
total_vids = len([f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")])
print(f"Total .mp4 files: {total_vids}")

## Step 3b: Download Split JSONs

The split files (which video IDs go to train/test) are small — download from HuggingFace.
Skip if already present.

In [ ]:
os.makedirs("msrvtt", exist_ok=True)

json_urls = {
    "msrvtt/msrvtt_train_7k.json": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_train_7k.json",
    "msrvtt/msrvtt_test_1k.json":  "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_test_1k.json",
}

for path, url in json_urls.items():
    if not os.path.exists(path):
        print(f"Downloading {path} ...")
        urllib.request.urlretrieve(url, path)
        print("  Done.")
    else:
        print(f"  Already exists: {path}")

## Step 3c: Diagnose Audio on a Few Videos

Quick sanity check — confirm audio streams exist in nyhuka/msrvtt before running the full patch.

In [ ]:
def probe_audio(video_path):
    """Returns (ffprobe_has_stream, wav_size_bytes, ffmpeg_rc)."""
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=codec_name",
        "-of", "default=noprint_wrappers=1",
        video_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    has_stream = bool(result.stdout.strip())

    tmp_wav = os.path.join(TMP_DIR, "probe_test.wav")
    cmd2 = f'ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{tmp_wav}"'
    res2 = subprocess.run(cmd2, shell=True, capture_output=True, text=True)
    wav_size = os.path.getsize(tmp_wav) if os.path.exists(tmp_wav) else 0
    if os.path.exists(tmp_wav):
        os.remove(tmp_wav)
    return has_stream, wav_size, res2.returncode


sample_vids = sorted([f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")])[:10]
print(f"{'Video':<30} {'ffprobe':>10} {'wav_bytes':>10} {'ffmpeg_rc':>10}")
print("-" * 64)
audio_found = 0
for v in sample_vids:
    has_s, wav_sz, rc = probe_audio(os.path.join(VIDEO_DIR, v))
    if has_s or wav_sz > 0:
        audio_found += 1
    print(f"{v:<30} {str(has_s):>10} {wav_sz:>10} {rc:>10}")

print(f"\nAudio found in {audio_found}/10 sampled videos.")
if audio_found == 0:
    print("WARNING: No audio detected — patching will not improve anything.")
else:
    print("Audio confirmed — proceed with patching.")

## Step 4: Copy V2 Feature Files to Working Dir

The existing V2 feature files need to be in `/kaggle/working/`.
Add them as a Kaggle dataset input and this cell copies them over.
If they're already in working dir (e.g. from a prior session), this is a no-op.

In [ ]:
# Adjust this to match the name of the dataset you uploaded the V2 features to
FEATURES_DATASET = "/kaggle/input/msrvtt-v2-features"  # <-- change if different

for fname in ["train_features_v2.pt", "test_features_v2.pt"]:
    dst = fname  # working dir
    if os.path.exists(dst):
        print(f"  {fname} already in working dir ({os.path.getsize(dst)/1e6:.1f} MB)")
        continue
    src = os.path.join(FEATURES_DATASET, fname)
    if os.path.exists(src):
        print(f"  Copying {fname} from dataset ...")
        shutil.copy2(src, dst)
        print(f"  Done ({os.path.getsize(dst)/1e6:.1f} MB)")
    else:
        print(f"  ERROR: {src} not found — check FEATURES_DATASET path above.")

## Step 5: Load Models

Only VGGish and ImageBind needed — CLIP is skipped (vision features not re-extracted).

In [ ]:
# VGGish
vggish = torch.hub.load("harritaylor/torchvggish", "vggish", trust_repo=True)
vggish.eval().to(device)
print("VGGish loaded.")

# ImageBind
ib_model = imagebind_model.imagebind_huge(pretrained=True)
ib_model.eval().to(device)
print("ImageBind loaded.")

## Step 6: Audio Extraction Helpers

In [ ]:
def extract_vggish_audio(video_path, tmp_name="vggish_audio.wav"):
    """Returns [128] VGGish embedding, or zeros if audio unavailable."""
    tmp = os.path.join(TMP_DIR, tmp_name)
    if os.path.exists(tmp):
        os.remove(tmp)
    cmd = f'ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{tmp}"'
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not os.path.exists(tmp) or os.path.getsize(tmp) < 1000:
        if os.path.exists(tmp):
            os.remove(tmp)
        return torch.zeros(128)
    try:
        with torch.no_grad():
            feat = vggish.forward(tmp)
            if feat.ndim > 1:
                feat = feat.mean(dim=0)
        os.remove(tmp)
        return feat.cpu()
    except Exception:
        if os.path.exists(tmp):
            os.remove(tmp)
        return torch.zeros(128)


def extract_imagebind_audio(video_path, tmp_name="ib_audio.wav"):
    """Returns [1024] ImageBind audio embedding, or None if extraction fails."""
    tmp = os.path.join(TMP_DIR, tmp_name)
    if os.path.exists(tmp):
        os.remove(tmp)
    cmd = f'ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{tmp}"'
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not os.path.exists(tmp) or os.path.getsize(tmp) < 1000:
        if os.path.exists(tmp):
            os.remove(tmp)
        return None
    try:
        audio_data = ib_data.load_and_transform_audio_data([tmp], device)
        inputs = {ModalityType.AUDIO: audio_data}
        with torch.no_grad():
            emb = ib_model(inputs)[ModalityType.AUDIO]
        os.remove(tmp)
        return emb.squeeze(0).cpu()  # [1024]
    except Exception as e:
        print(f"    IB audio error for {os.path.basename(video_path)}: {e}")
        if os.path.exists(tmp):
            os.remove(tmp)
        return None


def build_teacher(v_vis, v_aud):
    """Fuse IB vision + audio into normalized multimodal teacher."""
    if v_aud is None:
        return v_vis  # raw IB_vision, no normalization — same as original
    combined = (v_vis + v_aud) / 2.0
    return F.normalize(combined, dim=-1)  # unit-norm multimodal embedding


print("Helpers defined.")

## Step 7: Patch Function

Iterates over all videos in order, re-extracts audio, patches `z_aud`, `v_teacher`, `has_audio`.
Video filename is looked up via the split JSON (same mapping as original extraction).
Checkpoints every 200 videos.

In [ ]:
CHECKPOINT_EVERY = 200


def patch_features(features_pt, split_json, video_dir):
    """
    Loads an existing V2 feature file and patches z_aud, v_teacher, has_audio.
    Saves the result back to the same path (original backed up as *.orig.bak.pt).
    """
    ckpt_pt = features_pt.replace(".pt", "_patch_ckpt.pt")

    print(f"\n{'='*60}")
    print(f"Patching: {features_pt}")
    print(f"{'='*60}")

    # Load original features
    orig = torch.load(features_pt, map_location="cpu")
    v_teachers_orig = orig["v_teacher"].to(torch.float32)  # [N, 1024] = raw IB_vision
    video_ids_all   = list(orig["video_ids"])              # ordered list
    N = len(video_ids_all)
    print(f"Loaded {N} samples.")
    print(f"Current audio coverage: {orig['has_audio'].sum().item()}/{N}")

    # Build video_id -> filename map from split JSON
    with open(split_json) as f:
        jdata = json.load(f)
    vid_to_file = {item["video_id"]: item["video"] for item in jdata}

    # Check a few paths to confirm video_dir matches the JSON filenames
    sample_vid = video_ids_all[0]
    sample_file = vid_to_file.get(sample_vid, "UNKNOWN")
    sample_path = os.path.join(video_dir, sample_file)
    print(f"Path check: {sample_path}  -> exists={os.path.exists(sample_path)}")
    if not os.path.exists(sample_path):
        raise RuntimeError(
            f"Video not found: {sample_path}\n"
            f"Check that VIDEO_DIR ({video_dir}) contains the files listed in the JSON."
        )

    # Resume from checkpoint if it exists
    if os.path.exists(ckpt_pt):
        ckpt = torch.load(ckpt_pt, map_location="cpu")
        patched_z_aud     = list(ckpt["z_aud"])
        patched_v_teacher = list(ckpt["v_teacher"])
        patched_has_audio = list(ckpt["has_audio"])
        n_done = len(patched_z_aud)
        print(f"Resuming from checkpoint: {n_done}/{N} already patched.")
    else:
        patched_z_aud, patched_v_teacher, patched_has_audio = [], [], []
        n_done = 0

    remaining_ids = video_ids_all[n_done:]
    print(f"Videos left to process: {len(remaining_ids)}")

    n_audio_found = sum(h.item() for h in patched_has_audio)
    errors = 0

    for i, vid in enumerate(tqdm(remaining_ids, desc=os.path.basename(features_pt))):
        global_idx = n_done + i
        v_vis = v_teachers_orig[global_idx]  # stored raw IB_vision [1024]

        filename = vid_to_file.get(vid)
        if not filename:
            print(f"\n  Warning: {vid} not in JSON — keeping zero audio.")
            patched_z_aud.append(torch.zeros(128))
            patched_v_teacher.append(v_vis)
            patched_has_audio.append(torch.tensor(False))
            errors += 1
            continue

        video_path = os.path.join(video_dir, filename)
        if not os.path.exists(video_path):
            print(f"\n  Warning: {video_path} missing — keeping zero audio.")
            patched_z_aud.append(torch.zeros(128))
            patched_v_teacher.append(v_vis)
            patched_has_audio.append(torch.tensor(False))
            errors += 1
            continue

        try:
            z_aud  = extract_vggish_audio(video_path)
            v_aud  = extract_imagebind_audio(video_path)
            v_teach = build_teacher(v_vis, v_aud)
            audio_ok = (v_aud is not None)

            patched_z_aud.append(z_aud)
            patched_v_teacher.append(v_teach)
            patched_has_audio.append(torch.tensor(audio_ok, dtype=torch.bool))
            if audio_ok:
                n_audio_found += 1

        except Exception as e:
            print(f"\n  Error on {vid}: {e}")
            patched_z_aud.append(torch.zeros(128))
            patched_v_teacher.append(v_vis)
            patched_has_audio.append(torch.tensor(False))
            errors += 1

        if (i + 1) % CHECKPOINT_EVERY == 0:
            _save_ckpt(ckpt_pt, patched_z_aud, patched_v_teacher, patched_has_audio)
            print(f"  Checkpoint @ {len(patched_z_aud)}/{N} | "
                  f"audio={n_audio_found} | errors={errors}")

    # Assemble final patched dict — z_img and video_ids carried over unchanged
    patched = dict(orig)
    patched["z_aud"]     = torch.stack(patched_z_aud).to(torch.float32)
    patched["v_teacher"] = torch.stack(patched_v_teacher).to(torch.float32)
    patched["has_audio"] = torch.stack(patched_has_audio)

    # Backup original before overwriting
    bak = features_pt.replace(".pt", ".orig.bak.pt")
    if not os.path.exists(bak):
        shutil.copy2(features_pt, bak)
        print(f"  Backup saved: {bak}")

    torch.save(patched, features_pt)
    non_zero = (torch.norm(patched["z_aud"], dim=1) > 1e-5).sum().item()
    print(f"\nPatched file saved: {features_pt}")
    print(f"  z_aud      : {patched['z_aud'].shape}  non-zero={non_zero}/{N}")
    print(f"  v_teacher  : {patched['v_teacher'].shape}")
    print(f"  has_audio  : {patched['has_audio'].sum().item()}/{N} "
          f"({100*patched['has_audio'].float().mean():.1f}%)")
    print(f"  errors     : {errors}")

    if os.path.exists(ckpt_pt):
        os.remove(ckpt_pt)
        print("  Checkpoint removed.")


def _save_ckpt(path, z_auds, v_teachers, has_audios):
    torch.save({
        "z_aud":     torch.stack(z_auds),
        "v_teacher": torch.stack(v_teachers),
        "has_audio": torch.stack(has_audios),
    }, path)


print("Patch function defined.")

## Step 8: Run Patch

Test set first (~1000 videos, ~30 min on T4), then train (~7010 videos, ~2h on T4).
Only VGGish + IB audio — no vision re-extraction, roughly half the time of a full extraction run.

In [ ]:
patch_features(
    features_pt="test_features_v2.pt",
    split_json="msrvtt/msrvtt_test_1k.json",
    video_dir=VIDEO_DIR,
)

patch_features(
    features_pt="train_features_v2.pt",
    split_json="msrvtt/msrvtt_train_7k.json",
    video_dir=VIDEO_DIR,
)

## Step 9: Verify & Download

In [ ]:
from IPython.display import FileLink, display

for fname in ["test_features_v2.pt", "train_features_v2.pt"]:
    if not os.path.exists(fname):
        print(f"MISSING: {fname}")
        continue
    d = torch.load(fname, map_location="cpu")
    n_aud = d["has_audio"].sum().item()
    n_tot = len(d["video_ids"])
    non_zero_aud = (torch.norm(d["z_aud"].float(), dim=1) > 1e-5).sum().item()
    print(f"\n{fname}")
    print(f"  z_img      : {d['z_img'].shape}  (dtype: {d['z_img'].dtype})")
    print(f"  z_aud      : {d['z_aud'].shape}  non-zero={non_zero_aud}/{n_tot}")
    print(f"  v_teacher  : {d['v_teacher'].shape}")
    print(f"  has_audio  : {n_aud}/{n_tot} ({100*n_aud/n_tot:.1f}%)")
    display(FileLink(fname, result_html_prefix=f"Download {fname}: "))

print("\nAll .pt files in working dir:")
print([f for f in os.listdir(".") if f.endswith(".pt")])